<a href="https://colab.research.google.com/github/Grecia1225/data-science-projects/blob/main/News%20Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
with zipfile.ZipFile("/content/archive.zip", "r") as zip_ref:
    zip_ref.extractall("/content/")
print("Unzipped! Files ready!")

Unzipped! Files ready!


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
fake = pd.read_csv("/content/Fake.csv")
real = pd.read_csv("/content/True.csv")
fake["label"] = 0
real["label"] = 1
df = pd.concat([fake, real], ignore_index=True)
print(f"Total articles: {len(df)}")
print(f"Fake articles: {len(fake)}")
print(f"Real articles: {len(real)}")
print(df.head())

Total articles: 44898
Fake articles: 23481
Real articles: 21417
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29, 2017      0  
4  December 25, 2017      0  


In [ ]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["text"] = df["title"] + " " + df["text"]
df["text"] = df["text"].apply(clean_text)
df = df[["text", "label"]]
df = df.dropna()
print("Data cleaned!")
print(f"Total samples: {len(df)}")
print(df.head(2))

Data cleaned!
Total samples: 44898
                                                text  label
0  donald trump sends out embarrassing new years ...      0
1  drunk bragging trump staffer started russian c...      0


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print("Vectorization done!")

Training samples: 35918
Test samples: 8980
Vectorization done!


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
print("Model trained!")

Model trained!


In [ ]:
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

              precision    recall  f1-score   support

        Fake       0.99      0.99      0.99      4733
        Real       0.99      0.99      0.99      4247

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Overall Accuracy: 0.9889


In [ ]:
def predict_news(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    label = "REAL" if prediction == 1 else "FAKE"
    print(f"News: {text[:80]}...")
    print(f"Prediction: {label}\n")
predict_news("Scientists discover new vaccine that cures all diseases overnight")
predict_news("The President signed a new climate bill into law today")
predict_news("SHOCKING: Celebrities are secretly controlling the government!")

News: Scientists discover new vaccine that cures all diseases overnight...
Prediction: FAKE

News: The President signed a new climate bill into law today...
Prediction: FAKE

News: SHOCKING: Celebrities are secretly controlling the government!...
Prediction: FAKE



In [ ]:
predict_news("The Federal Reserve raised interest rates by 0.25 percent on Wednesday")
predict_news("Apple reported quarterly earnings of 90 billion dollars")
predict_news("ALIENS HAVE LANDED and the government is hiding it from us!!!")

News: The Federal Reserve raised interest rates by 0.25 percent on Wednesday...
Prediction: REAL

News: Apple reported quarterly earnings of 90 billion dollars...
Prediction: FAKE

News: ALIENS HAVE LANDED and the government is hiding it from us!!!...
Prediction: FAKE

